# Reference Resolution Data Exploration

In [1]:
import json
import pandas as pd
import pyarrow as pa
import os

import torch

from transformers import AutoTokenizer, AutoImageProcessor

from PIL import Image

import sys
import os.path as osp
import json
import pickle
import time
import itertools
import skimage.io as io
# import matplotlib.pyplot as plt
# from matplotlib.collections import PatchCollection
# from matplotlib.patches import Polygon, Rectangle
from pprint import pprint
import numpy as np
from refer import REFER

from tqdm import tqdm
from collections import defaultdict

from meter.datasets.base_dataset import BaseDataset
from meter.datamodules.datamodule_base import BaseDataModule
from meter.datasets.refcoco_dataset import RefcocoDataset

## Read in Raw Data

In [ ]:
class StrToBytes:
    def __init__(self, fileobj):
        self.fileobj = fileobj
    def read(self, size):
        return self.fileobj.read(size).encode()
    def readline(self, size=-1):
        return self.fileobj.readline(size).encode()

### Open Files

In [ ]:
data_root = '/home/claytonfields/nlp/code/data/coco'  # contains refclef, refcoco, refcoco+, refcocog and images
dataset = 'refcoco' 
splitBy = 'unc'

print ('loading dataset %s into memory...' % dataset)
print('testing')
# ROOT_DIR = osp.abspath(osp.dirname(__file__))
DATA_DIR = osp.join(data_root, dataset)
if dataset in ['refcoco', 'refcoco+', 'refcocog']:
    IMAGE_DIR = osp.join(data_root, 'images/mscoco/train2014')
elif dataset == 'refclef':
    IMAGE_DIR = osp.join(data_root, 'images/saiapr_tc-12')
else:
    print ('No refer dataset is called [%s]' % dataset)
    sys.exit()

# load refs from data/dataset/refs(dataset).json
tic = time.time()
ref_file = osp.join(DATA_DIR, 'refs('+splitBy+').p')
data = {}
data['dataset'] = dataset
data['refs'] = pickle.load(StrToBytes(open(ref_file, 'r')))

# load annotations from data/dataset/instances.json
instances_file = osp.join(DATA_DIR, 'instances.json')
instances = json.load(open(instances_file, 'r'))
data['images'] = instances['images']
data['annotations'] = instances['annotations']
data['categories'] = instances['categories']

# # create index
# createIndex()
# print ('DONE (t=%.2fs)' % (time.time()-tic)

### Aggregate Data

In [ ]:
Anns, Imgs, Cats, imgToAnns = {}, {}, {}, {}
for ann in data['annotations']:
    Anns[ann['id']] = ann
    imgToAnns[ann['image_id']] = imgToAnns.get(ann['image_id'], []) + [ann]
for img in data['images']:
    Imgs[img['id']] = img

In [ ]:
def process_ref(ref, path, Imgs, imgToAnns):
    img = Imgs[ref['image_id']]
    image_path = osp.join(path, img['file_name'])
    with open(image_path, "rb") as fp:
        binary = fp.read()

    # TODO: Process text
    sents = []
    for sent in ref['sentences']:
        sents.append(sent['sent'])
    
    anns= imgToAnns[ref['image_id']]
    ann_id = ref['ann_id']
    obj_ids = []
    bboxes = []
    for ann in anns:
        obj_ids.append(ann['id'])
        bboxes.append(ann['bbox'])
    label = obj_ids.index(ann_id)
        
    return [binary, sents, bboxes, label]

binary = process_ref(data['refs'][0], IMAGE_DIR, Imgs, imgToAnns)
binary[3]

In [ ]:
data['refs'][-1000]

### Write Data to .arrow File

In [ ]:
splits = ['train', 'val', 'test']

for split in splits:
    if split == 'test':
        refs = [ref for ref in tqdm(data['refs']) if 'test' in ref['split']]
    elif split == 'train' or split == 'val':
        refs = [ref for ref in tqdm(data['refs']) if ref['split'] == split]

    item_list = [process_ref(ref, IMAGE_DIR, Imgs, imgToAnns) for ref in refs]
    df = pd.DataFrame(item_list, columns=['image', 'sentences', 'bboxes', 'labels'])
    table = pa.Table.from_pandas(df)
    
    # os.makedirs(dataset_root, exist_ok=True)
    with pa.OSFile(
        f"refcoco_{splitBy}_{split}.arrow", "wb"
    ) as sink:
        with pa.RecordBatchFileWriter(sink, table.schema) as writer:
            writer.write_table(table)

## Read Data from .arrow File

In [ ]:
names = ['refcoco_unc_train']
text_column_name = 'sentences'
remove_duplicate = False
image_only = False

In [ ]:
if len(names) != 0:
    tables = [
        pa.ipc.RecordBatchFileReader(
            pa.memory_map(f"{name}.arrow", "r")
        ).read_all()
        for name in names
        if os.path.isfile(f"{name}.arrow")
    ]

    table_names = list()
    for i, name in enumerate(names):
        table_names += [name] * len(tables[i])

    table = pa.concat_tables(tables, promote=True)
    if text_column_name != "":
        text_column_name = text_column_name
        all_texts = table[text_column_name].to_pandas().tolist()
        if type(all_texts[0][0]) == str:
            all_texts = (
                [list(set(texts)) for texts in all_texts]
                if remove_duplicate
                else all_texts
            )
        else: #snli
            all_texts = (
                [[t[1].strip() for t in texts] for texts in all_texts]
            )
    else:
        all_texts = list()
else:
    all_texts = list()

In [ ]:
table.to_pandas()

In [ ]:
index_mapper = dict()

if text_column_name != "" and not image_only:
    j = 0
    for i, texts in enumerate(all_texts):
        for _j in range(len(texts)):
            index_mapper[j] = (i, _j)
            j += 1
else:
    for i in range(len(table)):
        index_mapper[i] = (i, None)

In [ ]:
index_mapper

### Extend Base Dataset

In [ ]:
class RefcocoDataset(BaseDataset):
    def __init__(self, *args, split="", max_bb = 42, **kwargs):
        assert split in ["train", "val", "test"]
        self.split = split
        self.max_bb = max_bb

        if split == "train":
            names = ['refcoco_unc_train']
        elif split == "val":
            # names = ["coco_caption_karpathy_val"]
            names = ['refcoco_unc_val']
        elif split == "test":
            names = ['refcoco_unc_test']

        super().__init__(*args, names=names, text_column_name="sentences", **kwargs)


    def __getitem__(self, index):
        max_bb = self.max_bb
        image_index, ref_index = self.index_mapper[index]
        label = self.table["labels"][image_index].as_py()
        image = np.array(self.get_raw_image(index))
        bboxes = self.table['bboxes'][image_index].as_py()
        sub_images = []
        for bbox in bboxes:
            bbox = [int(b) for b in bbox]
            sub = image[bbox[1]:bbox[1]+bbox[3],bbox[0]:bbox[0]+bbox[2]]
            if sub is not None:
                sub = self.processor(
                    sub, 
                    return_tensors='pt',
                    size={'height':self.image_size, 'width':self.image_size}
                )['pixel_values'][0]
                sub_images.append(sub.unsqueeze(0))
        num_sub_images = len(sub_images)
        num_pad = max_bb - num_sub_images 
        
        pad_image = torch.zeros(1,3,self.image_size,self.image_size)
        for _ in range(max_bb - num_sub_images):
            sub_images.append(pad_image)
        
        # text ids
        text = self.get_text(index)
        text_tokenized = text['text'][1]
        ids = text_tokenized['input_ids']
        repeat_ids = ids.repeat(num_sub_images,1)
        pad_ids =  torch.zeros(num_pad,self.max_text_len,dtype=torch.int8)
        text_ids = torch.cat((repeat_ids, pad_ids))#.to(torch.long)
        # text masks
        masks = text_tokenized['attention_mask']
        repeat_masks = masks.repeat(num_sub_images,1)
        pad_masks = torch.zeros(num_pad, self.max_text_len, dtype=torch.int8)
        text_masks = torch.cat((repeat_masks, pad_masks))#.to(torch.long)
        # text_labels
        labels = torch.full((self.max_text_len,),-100, dtype=torch.int8)
        repeat_labels = labels.repeat(num_sub_images, 1)
        pad_labels = torch.zeros(num_pad, self.max_text_len, dtype=torch.int8)
        text_labels = torch.cat((repeat_labels, pad_labels))#.to(torch.long)
        
        # target = self.table

        return_dict = {
            # 'ann_id' : ann_id,
            'image' : [torch.cat(sub_images)],#.to(self.device)],
            # 'obj_ids' : torch.tensor(obj_ids_total),#.to(self.device),
            'target' : label,#.to(self.device),
            'text' : text['text'][0],
            'text_ids' : text_ids,#.to(self.device),
            'text_labels' : text_labels,#.to(self.device),
            'text_masks' : text_masks,#.to(self.device)
        }
        
        return return_dict

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
processor = AutoImageProcessor.from_pretrained("facebook/deit-tiny-patch16-224")

In [ ]:
data_dir = '.'
transform_keys = ['imagenet']
image_size = 32
ds = RefcocoDataset(data_dir, transform_keys, image_size, tokenizer=tokenizer, processor=processor, split='train')

In [ ]:
ds[0]

### Prototype __get_item__()

In [ ]:
index = 0

max_bb = ds.max_bb
image_index, ref_index = ds.index_mapper[index]
print(image_index, ref_index)
label = ds.table["labels"][image_index].as_py()
image = np.array(ds.get_raw_image(index))
bboxes = ds.table['bboxes'][image_index].as_py()
sub_images = []
for bbox in bboxes:
    bbox = [int(b) for b in bbox]
    sub = image[bbox[1]:bbox[1]+bbox[3],bbox[0]:bbox[0]+bbox[2]]
    if sub is not None:
        sub = ds.processor(sub, return_tensors='pt')['pixel_values'][0]
        sub_images.append(sub.unsqueeze(0))
num_sub_images = len(sub_images)
num_pad = max_bb - num_sub_images 

pad_image = torch.zeros(1,3,224,224)
for _ in range(max_bb - num_sub_images):
    sub_images.append(pad_image)

# text ids
text_tokenized = ds.get_text(index)['text'][1]
ids = text_tokenized['input_ids']
repeat_ids = ids.repeat(num_sub_images,1)
pad_ids =  torch.zeros(num_pad,ds.max_text_len,dtype=torch.int8)
text_ids = torch.cat((repeat_ids, pad_ids))#.to(torch.long)
# text masks
masks = text_tokenized['attention_mask']
repeat_masks = masks.repeat(num_sub_images,1)
pad_masks = torch.zeros(num_pad, ds.max_text_len, dtype=torch.int8)
text_masks = torch.cat((repeat_masks, pad_masks))#.to(torch.long)
# text_labels
labels = torch.full((ds.max_text_len,),-100, dtype=torch.int8)
repeat_labels = labels.repeat(num_sub_images, 1)
pad_labels = torch.zeros(num_pad, ds.max_text_len, dtype=torch.int8)
text_labels = torch.cat((repeat_labels, pad_labels))#.to(torch.long)
text_labels

In [ ]:
ds.table.to_pandas()

## DataModule

In [ ]:
config = {  
    "exp_name":"finetune_mrpc",
    "seed" : 42,
    # "datasets" : ["coco", "vg", "sbu", "gcc"],
    "datasets" : ["coco", "vg"],
    # "datasets" : ["coco"],
    "loss_names" : {'itm': 0,
    'mlm': 0,
    'mpp': 0,
    'vqa': 0,
    'vcr': 0,
    'vcr_qar': 0,
    'nlvr2': 0,
    'irtr': 0,
    'contras': 0,
    'snli': 0,
    'ref': 0,
    'mrpc': 0,
    'rte' : 0,
    'wnli': 0,
    'sst2' : 0,
    'qqp' : 0,
    'qnli' : 0,
    'mnli' : 0,
    'cola' : 1
    },
    "batch_size" : 32,  # this is a desired batch size; pl trainer will accumulate gradients when per step batch is smaller.

    # Image setting
    "image_encoder" : "facebook/deit-tiny-patch16-224",
    "random_init_vision_encoder" : False,
    "image_encoder_hidden_size" : 192,
    "image_size" : 224,
    "patch_size" : 16,
    "draw_false_image" : 1,
    "image_only" : False,
    "resolution_before" : 224,
    "train_transform_keys" : ["imagenet"],
    "val_transform_keys" : ["imagenet"],

    # Text Setting
    "text_encoder" : "google/electra-small-discriminator",
    "random_init_text_encoder" : False,
    "text_encoder_hidden_size" : 256,
    "vocab_size" : 30522,
    "whole_word_masking" : False, # note that whole_word_masking does not work for RoBERTa
    "mlm_prob" : 0.15,
    "draw_false_text" : 0,
    "vqav2_label_size" : 3129,
    "max_text_len" : 128,

    # CrossLayer Setting
    "num_cross_layers" : 6,
    "cross_layer_hidden_size" : 256,
    "num_cross_layer_heads" : 4,
    "cross_layer_mlp_ratio" : 4,
    "cross_layer_drop_rate" : 0.1,
    
    # Architecture Setting
    "two_tower" : False,
    "multi_modal_encoder" : 'dandelin/vilt-b32-mlm',
    
    
    
    # Optimizer Setting
    "optim_type" : "adamw",
    "learning_rate" : 5e-5,
    "weight_decay" : 0.0,
    "decay_power" : 1,
    "max_epoch" : 3,
    "max_steps" : 100000,
    "warmup_steps" : 0,
    "end_lr" : 0,
    "lr_mult_head" : 5,  # multiply lr for downstream heads
    "lr_mult_cross_modal" : 5,  # multiply lr for the cross-modal module

    # Encoder Settings
    "freeze_image_encoder" : True,
    "freeze_text_encoder" : False,
    'freeze_cross_modal_layers' : True,
    
    'text_only' : False,
    

    # Downstream Setting
    "get_recall_metric" : False,
    
    'freeze' : True,
    
    "model_type" : "METER",

    # PL Trainer Setting
    "resume_from" : None,
    "fast_dev_run" : False,
    "val_check_interval" : 1.0,
    "test_only" : False,

    "data_root" : "/home/claytonfields/nlp/code/meter/data/arrow",
    "log_dir" : "result",
    "per_gpu_batchsize" : 32,  # you should define this manually with per_gpu_batch_size:#
    "num_gpus" : 1,
    "num_nodes" : 1,
    "load_path" : "/home/claytonfields/nlp/code/meter/result/mlm_itm_seed0_from_/meter_electra_small_deit_tiny_p16_is224_bs288_is1M/checkpoints/epoch=43-step=898039.ckpt",
    # "load_path" : '/home/claytonfields/nlp/code/meter/result/mlm_itm_deit_fr_electra_fr_is224_ps16_bs336_pgbs84_ts100k/checkpoints/epoch=5-step=96215.ckpt',
    "num_workers" : 12,
    "precision" : 32
}

In [ ]:
class RefcocoDataModule(BaseDataModule):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    @property
    def dataset_cls(self):
        return RefcocoDataset

    @property
    def dataset_cls_no_false(self):
        return RefcocoDataset

    @property
    def dataset_name(self):
        return "refcoco"

In [ ]:
dm = RefcocoDataModule(config)
dm.setup('train')
dm.prepare_data()
dl = dm.test_dataloader()
next(iter(dl))

In [ ]:
ds.table[0]

In [ ]:
df = ds.table.to_pandas()

In [ ]:
df.info()

In [ ]:
def type_mapping(dtype):
    if dtype=='object':
        return "string[pyarrow]"
    elif dtype=='int64':
        return "uint64[pyarrow]"

In [ ]:
ds.table.to_pandas(types_mapper=type_mapping)

In [ ]:
df[len(df['bboxes'])<=42]

In [ ]:
df['bboxes'][0].size

In [2]:
def check_len(item):
    return item.size <= 42

In [ ]:
sub = df[df['bboxes'].apply(check_len)]
sub.reset_index(inplace=True, drop=True)
# sub.drop(columns=['index'], inplace=True)
sub

In [ ]:
table = pa.Table.from_pandas(sub)

In [ ]:
table.to_pandas()

In [ ]:
sub

In [ ]:
temp_train = pd.read_parquet('/home/claytonfields/nlp/code/meter/data/arrow/refcoco_unc_train.parquet', engine='pyarrow', dtype_backend='pyarrow')
temp_train

In [ ]:
temp_train.info()

In [2]:
import time

In [6]:
start = time.time()
df = pd.read_parquet('/home/claytonfields/nlp/code/meter/data/arrow/refcoco_unc_train.parquet', engine='pyarrow', dtype_backend='pyarrow')
# sub = df.where(df['bboxes'].apply(check_len))
df[df['bboxes'].apply(check_len)]
sub.reset_index(inplace=True, drop=True)
end = time.time()
end - start

ArrowInvalid: offset overflow while concatenating arrays

In [4]:
start = time.time()
df = pd.read_parquet('/home/claytonfields/nlp/code/meter/data/arrow/refcoco_unc_train.parquet', engine='pyarrow')
sub = df[df['bboxes'].apply(check_len)]
sub.reset_index(inplace=True, drop=True)
end = time.time()
end - start

11.72469162940979

In [5]:
start = time.time()
df = pd.read_parquet('/home/claytonfields/nlp/code/meter/data/arrow/refcoco_unc_train.parquet')
sub = df[df['bboxes'].apply(check_len)]
sub.reset_index(inplace=True, drop=True)
end = time.time()
end - start

12.911767959594727